# Substitution Matrix Evaluation

This notebook evaluates amino acid substitution matrices based on their correlations with biochemical properties and codon distances.

The correlations are reported descriptively, without significance testing. The 190 pairs are built from 20 amino acids, each of which enters 19 of them, so they are not the independent observations a Spearman p-value assumes. The committed ranking is produced by `scripts/build_matrix_ranking.py`, which reproduces every committed row before it is allowed to write.

**Author:** KSS Project  
**Date:** 2025-01-19

## 1. Imports and Configuration

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# Add src to path for main application modules
sys.path.insert(0, '../scripts')
sys.path.append('../src')
from mutation_score import get_mutational_scores, load_substitution_matrix

# Import evaluation utilities from local module
from amino_acid_utils import (
    CODON_TABLE,
    AMINO_ACIDS,
    AA_3_TO_1,
    compute_codon_distance,
    get_min_codon_distance,
    calculate_property_distances,
    generate_amino_acid_pairs,
    precompute_codon_distances
)

## 2. Configuration Parameters

In [2]:
# Configuration
MATRICES_DIR = "../src/substitution_matrices"
PROPERTIES_FILE = '../scripts/amino_acid_properties.csv'
OUTPUT_DIR = "../results/matrix_evaluation"
TOP_N = 10

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Genetic Code and Amino Acid Data

In [3]:
# Use standard genetic code from amino_acid_utils
print(f"Amino acids: {len(AMINO_ACIDS)}")
print(f"Amino acids: {AMINO_ACIDS}")

Amino acids: 20
Amino acids: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']


## 4. Helper Functions

In [4]:
def format_correlation(corr):
    """Format a correlation for the ranking table."""
    return f"{corr:.3f}"

## 5. Load Amino Acid Properties

In [5]:
# Load amino acid properties using mapping from amino_acid_utils
df_properties = pd.read_csv(PROPERTIES_FILE)
df_properties['AA_1letter'] = df_properties['AA'].map(AA_3_TO_1)
df_properties = df_properties.set_index('AA_1letter')
df_properties = df_properties.drop('AA', axis=1)

# Display properties
print("Amino acid properties:")
display(df_properties)

# Convert to dictionary format (using 1-letter codes)
amino_acid_properties = {col: df_properties[col].to_dict() for col in df_properties.columns}
property_names = list(amino_acid_properties.keys())

Amino acid properties:


,Hydrophob.,Polarity,Volume,Area Lost,pI,In-Out,Aroma.
AA_1letter,,,,,,,
A,0.31,8.1,27.5,0.74,6.00,0.20,0.025
R,-1.01,10.5,105.0,0.64,10.76,-1.34,0.750
N,-0.60,11.6,58.7,0.63,5.41,-0.69,0.200
D,-0.77,13.0,40.0,0.62,2.77,-0.72,-0.025
C,1.54,5.5,44.6,0.91,5.05,0.67,0.200
Q,-0.22,10.5,80.7,0.62,5.65,-0.74,0.300
E,-0.64,12.3,62.0,0.62,3.22,-1.09,0.050
G,0.00,9.0,53.2,0.72,5.97,0.06,0.000
H,0.13,10.4,79.0,0.78,7.59,0.04,0.450


## 6. Compute Reference Distances

In [6]:
# Generate all amino acid pairs using utility function
pairs = generate_amino_acid_pairs()
print(f"Total amino acid pairs: {len(pairs)}")

# Compute minimum codon distances using utility function
min_codon_distances = precompute_codon_distances()

# Display distribution
codon_dist_values = list(min_codon_distances.values())
print(f"\nMinimum codon distance distribution:")
print(pd.Series(codon_dist_values).value_counts().sort_index())

Total amino acid pairs: 190

Minimum codon distance distribution:
1     75
2    101
3     14
Name: count, dtype: int64


## 7. Load Available Matrices

In [7]:
# Find all matrix files
matrix_files = {}

for file in os.listdir(MATRICES_DIR):
    if file.endswith('.json'):
        matrix_files[file[:-5]] = os.path.join(MATRICES_DIR, file)

matrix_names = sorted(matrix_files.keys())
print(f"Available matrices: {len(matrix_names)}")
for name in matrix_names:
    print(f"  - {name}")


Available matrices: 38
  - BENNER22
  - BENNER6
  - BENNER74
  - BLASTP
  - BLOSUM45
  - BLOSUM50
  - BLOSUM62
  - BLOSUM80
  - BLOSUM90
  - DAYHOFF
  - FENG
  - FLAVI
  - FLU
  - GENETIC
  - GONNET1992
  - GRANTHAM
  - HIVb
  - HIVw
  - JOHNSON
  - JONES
  - LEVIN
  - MCLACHLAN
  - MDM78
  - MIYATA
  - MIYATA_EVO
  - PAM120
  - PAM250
  - PAM30
  - PAM70
  - RAO
  - RISLER
  - SNEATH
  - STR
  - VTML160
  - VTML200
  - VTML250
  - WAG
  - rtREV


## 8. Evaluate Matrices

In [8]:
def evaluate_matrix(matrix_name):
    """Evaluate a single substitution matrix."""
    # Get mutation scores
    scores = get_mutational_scores(pairs, substitution_matrix_type=matrix_name)
    scores_array = np.array([scores[p] for p in pairs])
    
    # Calculate correlations with properties
    correlations = {}
    
    for prop_name in property_names:
        prop_distances = calculate_property_distances(amino_acid_properties[prop_name], pairs)
        correlations[prop_name] = stats.spearmanr(prop_distances, scores_array).statistic
    
    # Calculate correlation with codon distances
    codon_dist_array = np.array([min_codon_distances[p] for p in pairs])
    correlations['MinCodD'] = stats.spearmanr(codon_dist_array, scores_array).statistic
    
    # Composite score
    composite_score = sum(correlations.values())
    
    return correlations, composite_score


# Evaluate all matrices
results = {}
print("Evaluating matrices...\n")

for i, matrix_name in enumerate(matrix_names, 1):
    try:
        correlations, composite = evaluate_matrix(matrix_name)
        results[matrix_name] = {
            'correlations': correlations,
            'composite_score': composite
        }
        print(f"[{i}/{len(matrix_names)}] {matrix_name}: {composite:.3f}")
    except Exception as e:
        print(f"[{i}/{len(matrix_names)}] {matrix_name}: ERROR - {e}")

print(f"\nSuccessfully evaluated: {len(results)}/{len(matrix_names)}")

Evaluating matrices...

[1/38] BENNER22: 3.439
[2/38] BENNER6: 2.982
[3/38] BENNER74: 3.441
[4/38] BLASTP: 3.187
[5/38] BLOSUM45: 2.953
[6/38] BLOSUM50: 3.142
[7/38] BLOSUM62: 3.271
[8/38] BLOSUM80: 3.227
[9/38] BLOSUM90: 3.202
[10/38] DAYHOFF: 3.012
[11/38] FENG: 3.525
[12/38] FLAVI: 2.623


[13/38] FLU: 2.762
[14/38] GENETIC: 2.158
[15/38] GONNET1992: 3.402
[16/38] GRANTHAM: 2.967


[17/38] HIVb: 3.056
[18/38] HIVw: 2.171


[19/38] JOHNSON: 2.878


[20/38] JONES: 3.309
[21/38] LEVIN: 2.787
[22/38] MCLACHLAN: 2.889
[23/38] MDM78: 3.054
[24/38] MIYATA: 3.566
[25/38] MIYATA_EVO: 4.578
[26/38] PAM120: 2.914
[27/38] PAM250: 3.013
[28/38] PAM30: 2.890
[29/38] PAM70: 2.935
[30/38] RAO: 2.480
[31/38] RISLER: 1.125
[32/38] SNEATH: 2.373


[33/38] STR: 2.650
[34/38] VTML160: 3.257
[35/38] VTML200: 3.319
[36/38] VTML250: 3.351


[37/38] WAG: 3.291
[38/38] rtREV: 3.449

Successfully evaluated: 38/38


## 9. Create Results Table

In [9]:
# Sort by composite score
sorted_matrices = sorted(results.items(), key=lambda x: x[1]['composite_score'], reverse=True)

# Build results dataframe
rows = []
for rank, (matrix_name, data) in enumerate(sorted_matrices, 1):
    row = {'Rank': rank, 'Matrix': matrix_name}
    
    # Add formatted correlations
    for prop in property_names + ['MinCodD']:
        row[prop] = format_correlation(data['correlations'][prop])
    
    row['Composite'] = round(data['composite_score'], 3)
    rows.append(row)

df_results = pd.DataFrame(rows)

# Display top N
print(f"\n{'='*120}")
print(f"SUBSTITUTION MATRICES RANKING")
print(f"{'='*120}\n")
display(df_results.head(len(df_results)).style.hide(axis='index'))


SUBSTITUTION MATRICES RANKING



Rank,Matrix,Hydrophob.,Polarity,Volume,Area Lost,pI,In-Out,Aroma.,MinCodD,Composite
1,MIYATA_EVO,0.849,0.791,0.286,0.764,0.419,0.773,0.284,0.412,4.578000
2,MIYATA,0.738,0.727,0.398,0.554,0.130,0.471,0.277,0.270,3.566000
3,FENG,0.617,0.628,0.216,0.524,0.163,0.416,0.253,0.707,3.525000
4,rtREV,0.709,0.682,0.280,0.622,0.055,0.501,0.287,0.312,3.449000
5,BENNER74,0.697,0.666,0.302,0.621,0.006,0.433,0.289,0.427,3.441000
6,BENNER22,0.674,0.638,0.283,0.550,0.031,0.397,0.321,0.544,3.439000
7,GONNET1992,0.705,0.670,0.295,0.629,-0.001,0.440,0.276,0.388,3.402000
8,VTML250,0.677,0.638,0.311,0.595,0.021,0.438,0.302,0.370,3.351000
9,VTML200,0.664,0.630,0.323,0.575,0.019,0.416,0.309,0.383,3.319000
10,JONES,0.626,0.600,0.245,0.503,0.043,0.389,0.303,0.601,3.309000


## 10. Best Matrix Details

In [10]:
# Get best matrix
best_matrix_name = sorted_matrices[0][0]
best_data = sorted_matrices[0][1]

print(f"Best performing matrix: {best_matrix_name}")
print(f"Composite score: {best_data['composite_score']:.3f}\n")

# Detailed correlations
print("Detailed correlations:\n")
detail_rows = []
for prop in property_names + ['MinCodD']:
    detail_rows.append({
        'Property': prop,
        'Correlation': round(best_data['correlations'][prop], 4)
    })

df_best = pd.DataFrame(detail_rows)
display(df_best)

Best performing matrix: MIYATA_EVO
Composite score: 4.578

Detailed correlations:



,Property,Correlation
0,Hydrophob.,0.8489
1,Polarity,0.7913
2,Volume,0.2860
3,Area Lost,0.7643
4,pI,0.4186
5,In-Out,0.7729
6,Aroma.,0.2835
7,MinCodD,0.4124


## 11. Save Results

In [11]:
# Save top N matrices
top_file = os.path.join(OUTPUT_DIR, f'substitution_matrices_ranking_top_{TOP_N}.csv')
df_results.head(TOP_N).to_csv(top_file, index=False)
print(f"Top {TOP_N} results saved to: {top_file}")

# Save complete ranking
complete_file = os.path.join(OUTPUT_DIR, 'substitution_matrices_ranking_complete.csv')
df_results.to_csv(complete_file, index=False)
print(f"Complete results saved to: {complete_file}")

# Save best matrix details
best_file = os.path.join(OUTPUT_DIR, f'{best_matrix_name}_detailed_correlations.csv')
df_best.to_csv(best_file, index=False)
print(f"Best matrix details saved to: {best_file}")

Top 10 results saved to: ../results/matrix_evaluation\substitution_matrices_ranking_top_10.csv
Complete results saved to: ../results/matrix_evaluation\substitution_matrices_ranking_complete.csv
Best matrix details saved to: ../results/matrix_evaluation\MIYATA_EVO_detailed_correlations.csv


## 12. Summary Statistics

In [12]:
print("\nSUMMARY")
print("="*50)
print(f"Total matrices evaluated: {len(results)}")
print(f"Amino acid pairs analyzed: {len(pairs)}")
print(f"Properties analyzed: {len(property_names) + 1}")  # +1 for codon distance
print(f"\nBest matrix: {best_matrix_name}")
print(f"Best composite score: {best_data['composite_score']:.3f}")


SUMMARY
Total matrices evaluated: 38
Amino acid pairs analyzed: 190
Properties analyzed: 8

Best matrix: MIYATA_EVO
Best composite score: 4.578
